# 🎓 Capstone Project: Forecasting Temperature with ANN & LSTM
**Dataset:** Jena Climate 2009–2016 (Max Planck Institute, Germany) — weather readings every 10 minutes, 14 features, ~420,000 rows
**Total time:** 6 hours | **Total marks:** 100 | **Framework:** TensorFlow / Keras

---
### 📌 Problem Statement
You are a data scientist at a weather-analytics company. Your manager wants to know:
> *"Given the past **5 days (120 hours)** of weather readings, can we predict the **temperature 24 hours from now**, and is a sequence model (LSTM) actually better than a simple feed-forward network (ANN)?"*

### 🕒 Time Plan
| Part | Topic | Time | Marks |
|---|---|---|---|
| 0 | Setup & Data Loading | 20 min | 0 |
| 1 | Exploratory Data Analysis | 60 min | 20 |
| 2 | Preprocessing & Windowing | 40 min | 10 |
| 3 | Baselines | 15 min | 5 |
| 4 | ANN Model | 50 min | 15 |
| 5 | LSTM Model | 70 min | 20 |
| 6 | Optimizer & Learning-Rate Study | 45 min | 10 |
| 7 | Final Evaluation & Error Analysis | 35 min | 10 |
| 8 | Report / Conclusion | 25 min | 10 |

### ✅ Rules
- Answer in the cells marked `# YOUR CODE HERE`. Write your **observations in the markdown cells** — code without interpretation gets half marks.
- Set random seeds (given below) so results are reproducible.
- **Never** shuffle the data before splitting — this is a time series (data leakage = heavy penalty).
- Keep a **results table** (Part 7) — every model must be compared on the same test set.

## Part 0 — Setup & Data Loading 

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import os, random, time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
sns.set_style("whitegrid")
print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Download dataset (works in Colab / Kaggle / local with internet)
zip_path = keras.utils.get_file(
    origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip",
    fname="jena_climate_2009_2016.csv.zip", extract=True)
csv_path = zip_path.replace(".zip", "") if zip_path.endswith(".zip") else zip_path
if not os.path.exists(csv_path):
    csv_path = os.path.join(os.path.dirname(zip_path), "jena_climate_2009_2016.csv")
df = pd.read_csv(csv_path)
df.head()

> 💡 **Alternative (Kaggle):** search Kaggle for *"Jena Climate"* and load the CSV from `/kaggle/input/...` if internet is blocked.

## Part 1 — Exploratory Data Analysis ( 20 marks)

**Q1.1 (3 marks) – Data understanding.** Print shape, dtypes, `describe()`, number of missing values and duplicate rows. Convert `Date Time` to datetime (format `%d.%m.%Y %H:%M:%S`) and set it as index. What is the time range and sampling interval? Is the interval consistent everywhere?

In [ ]:
# YOUR CODE HERE

**Observations (write here):**

**Q1.2 (3 marks) – Data quality.** Look at the `wv (m/s)` and `max. wv (m/s)` columns. Do you notice impossible values (e.g., -9999)? Find them, explain why they are errors, and decide how to fix them (replace with 0? NaN + interpolation?). **Justify your choice.** Apply the fix to `df`.

In [ ]:
# YOUR CODE HERE

**Observations / justification:**

**Q1.3 (3 marks) – Resampling.** 10-minute data is too fine for a 5-day window (720 steps!). Resample to **hourly** data (take every 6th row or use `.resample('1H').mean()`). Report the new shape. Why does this help the model and the training time?

In [ ]:
# YOUR CODE HERE

**Answer:**

**Q1.4 (4 marks) – Target & distributions.** (a) Plot the temperature `T (degC)` over the whole period. (b) Plot histograms/KDEs of **all** features in a grid. (c) Draw box plots to detect outliers. Which features are skewed? Which have outliers that are *real* (weather extremes) vs *errors*?

In [ ]:
# YOUR CODE HERE

**Observations:**

**Q1.5 (4 marks) – Time patterns.** (a) Plot average temperature **by month** and **by hour of day**. (b) Zoom into any 2-week window and plot it. (c) Perform a seasonal decomposition (`statsmodels.tsa.seasonal.seasonal_decompose`, period=24 on a one-month slice) or plot the autocorrelation (`pandas.plotting.autocorrelation_plot`). What repeating cycles do you find? What does this tell you about how long a window the model needs?

In [ ]:
# YOUR CODE HERE

**Observations:**

**Q1.6 (3 marks) – Relationships.** Plot a correlation heatmap. Which features are almost perfectly correlated with temperature (e.g., `Tpot (K)`)? Is it **valid** to keep a feature that is basically the target in a different unit? Which features would you drop and why? Also create **cyclical time features**: `hour_sin, hour_cos, month_sin, month_cos` (hint: `np.sin(2*np.pi*hour/24)`) and explain why sin/cos is better than raw hour numbers.

In [ ]:
# YOUR CODE HERE

**Observations:**

## Part 2 — Preprocessing & Windowing (10 marks)

**Q2.1 (2 marks) – Chronological split.** Split the hourly data **70% train / 15% validation / 15% test** *in time order* (no shuffling). Print the date ranges of each split.

In [ ]:
# YOUR CODE HERE

**Q2.2 (2 marks) – Scaling without leakage.** Fit `StandardScaler` on the **train set only**, then transform val and test. Explain in one line what leakage would happen if you fitted it on the whole dataset. Keep a separate scaler (or the mean/std) for the target so you can inverse-transform predictions to °C.

In [ ]:
# YOUR CODE HERE

**Why train-only?**

**Q2.3 (4 marks) – Build windows.** Write a function `make_windows(data, target_idx, past=120, horizon=24)` that returns:
- `X` of shape `(samples, 120, n_features)` — the past 120 hours
- `y` of shape `(samples,)` — the temperature **24 hours after** the last hour of the window

Apply it to train/val/test. Print all shapes. (You may also use `tf.keras.utils.timeseries_dataset_from_array`.) **Draw/describe one window by hand to prove your indexing is not off by one.**

In [ ]:
PAST, HORIZON = 120, 24
def make_windows(data, target_idx, past=PAST, horizon=HORIZON):
    # YOUR CODE HERE
    pass

**Q2.4 (2 marks) – tf.data pipeline.** Wrap the arrays in `tf.data.Dataset` with `batch(64)` and `prefetch(tf.data.AUTOTUNE)`. Shuffle **only the training windows** (shuffling windows is fine; shuffling the raw series is not). Why is that distinction valid?

In [ ]:
# YOUR CODE HERE

## Part 3 — Baselines (5 marks)
A deep model is only useful if it beats something simple.

**Q3.1 (3 marks)** Compute test-set **MAE and RMSE (in °C)** for two baselines:
1. *Persistence*: predict that temperature 24 h from now = temperature right now.
2. *Linear Regression* (or Ridge) on the flattened window `(samples, 120*features)`.

**Q3.2 (2 marks)** Why is persistence a surprisingly strong baseline for weather? Which one should the neural networks beat?

In [ ]:
def evaluate(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name:25s} MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}")
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

results = []   # keep appending dicts here; you'll make a table in Part 7
# YOUR CODE HERE

**Answer:**

## Part 4 — ANN (Feed-Forward) Model (15 marks)

**Q4.1 (4 marks) – Architecture.** Build a function `build_ann(input_shape, units=(128,64,32), dropout=0.2, l2=1e-4)` with `Flatten` → `Dense` layers (ReLU, with `BatchNormalization` optional) → `Dropout` → `Dense(1)` output (linear). Print `model.summary()`. Explain: why linear activation at the output? Why MSE/Huber loss for regression?

In [ ]:
def build_ann(input_shape, units=(128, 64, 32), dropout=0.2, l2=1e-4):
    # YOUR CODE HERE
    pass

**Answer:**

**Q4.2 (5 marks) – Training with callbacks.** Compile with `Adam(learning_rate=1e-3)`, loss `mse`, metric `mae`. Train for up to **100 epochs** using **all three** callbacks:
- `EarlyStopping()`
- `ModelCheckpoint()`
- `ReduceLROnPlateau()`

Answer: At which epoch did training stop? Which epoch's weights were saved? What does `restore_best_weights=True` do and what happens if you set it to `False`?

In [ ]:
# YOUR CODE HERE

**Answer:**

**Q4.3 (3 marks) – Learning curves.** Plot training vs. validation loss (and MAE). Diagnose: *underfit, good fit or overfit?* Mark the epoch where the validation loss is lowest with a vertical line.

In [ ]:
# YOUR CODE HERE

**Diagnosis:**

**Q4.4 (3 marks) – Load & evaluate.** Load `best_ann.keras` with `keras.models.load_model`, predict on the test set, **inverse-transform to °C**, and compute MAE / RMSE / R². Append to `results`. Plot predicted vs actual for the first 300 test samples.

In [ ]:
# YOUR CODE HERE

## Part 5 — LSTM Model (20 marks)

**Q5.1 (4 marks) – Architecture.** Build `build_lstm(input_shape, units=(64,32), dropout=0.2, recurrent_dropout=0.0)` with stacked `LSTM` layers (`return_sequences=True` for all but the last), `Dropout`, and a `Dense(1)` output. Print the summary.
Explain: (a) why `return_sequences=True` for the first layer? (b) why is `recurrent_dropout>0` slow on GPU? (c) Compare the number of parameters with your ANN.

In [ ]:
def build_lstm(input_shape, units=(64, 32), dropout=0.2, recurrent_dropout=0.0):
    # YOUR CODE HERE
    pass

**Answers:**

**Q5.2 (5 marks) – Train.** Train the LSTM with the **same three callbacks** (checkpoint saved as `best_lstm.keras`), the same optimizer settings as the ANN, and `clipnorm=1.0` in Adam. Record the **training time per epoch**. Why is gradient clipping commonly used for RNNs? (Mention *exploding gradients*.)

In [ ]:
# YOUR CODE HERE

**Answer:**

**Q5.3 (3 marks) – Evaluate.** Same evaluation as the ANN (°C, MAE/RMSE/R²) — append to `results`. Plot learning curves and predicted vs. actual.

In [ ]:
# YOUR CODE HERE

**Q5.4 (4 marks) – Variants.** Try **one** of the following and compare with the baseline LSTM (use the same callbacks):
- (a) `GRU` instead of `LSTM`
- (b) `Bidirectional(LSTM)` — *discuss whether bidirectional makes sense for forecasting*
- (c) `Conv1D` front-end followed by LSTM
- (d) Predict the full next 24 hours (multi-step output) instead of one value

Add the result to `results`.

In [ ]:
# YOUR CODE HERE

**Discussion:**

**Q5.5 (4 marks) – ANN vs LSTM.** Write a comparison (5–8 lines): accuracy, training time, parameters, overfitting behavior. Did the LSTM actually win? If not, give **two possible reasons** (e.g., window too long, small signal beyond seasonality, insufficient tuning).

**Comparison:**

## Part 6 — Optimizer & Learning-Rate Study (10 marks)

**Q6.1 (6 marks)** Using the **better of your ANN/LSTM** architectures, train the *same model from the same initial weights* with each optimizer (max 40 epochs, EarlyStopping patience=6):

| Optimizer | Settings |
|---|---|
| SGD | lr=0.01 |
| SGD + Momentum | lr=0.01, momentum=0.9 |
| RMSprop | lr=1e-3 |
| Adam | lr=1e-3 |
| AdamW | lr=1e-3, weight_decay=1e-4 |

Plot **validation loss vs epoch for all optimizers on one chart** and make a table: *best val loss, epoch of best, time per epoch*.

In [ ]:
optimizer_configs = {
    "SGD":          lambda: optimizers.SGD(learning_rate=0.01),
    "SGD+Momentum": lambda: optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop":      lambda: optimizers.RMSprop(learning_rate=1e-3),
    "Adam":         lambda: optimizers.Adam(learning_rate=1e-3),
    "AdamW":        lambda: optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
}
# YOUR CODE HERE  (hint: re-seed with tf.keras.utils.set_random_seed(SEED) before building each model)

**Q6.2 (2 marks) – Learning-rate sensitivity.** Train with Adam at learning rates `[1e-4, 1e-3, 1e-2, 1e-1]`. Plot the results. What happens at 1e-1 and at 1e-4?

In [ ]:
# YOUR CODE HERE

**Observations:**

**Q6.3 (2 marks) – Written explanation.** In your own words (max 6 lines): why does plain SGD converge slower than Adam here? What does *momentum* add? What do `ReduceLROnPlateau` and a learning-rate schedule (e.g., `CosineDecay`) do?

**Answer:**

## Part 7 — Final Evaluation & Error Analysis (10 marks)

**Q7.1 (3 marks) – Results table.** Build a DataFrame from `results` with MAE, RMSE, R² of: Persistence, Linear Regression, ANN, LSTM, your variant. Highlight the best row. Plot a bar chart of MAE.

In [ ]:
# YOUR CODE HERE

**Q7.2 (4 marks) – Error analysis.** For your best model: (a) plot residuals (histogram + residuals vs time), (b) compute MAE **per month** or **per season** — when does the model fail most? (c) show the 3 worst predictions and explain what happened in the weather around that time.

In [ ]:
# YOUR CODE HERE

**Findings:**

**Q7.3 (3 marks) – Feature importance (permutation).** For the best model, shuffle **one feature at a time** across the test windows and measure the increase in MAE. Plot the top features. Does it match your EDA intuition?

In [ ]:
# YOUR CODE HERE

**Findings:**

## Part 8 — Report & Conclusion (10 marks)

**Q8.1 (5 marks) – Executive summary** (150–200 words) for your manager: the problem, the best model, its error in °C, and whether the complexity of an LSTM is worth it.

**Q8.2 (3 marks) – Limitations & next steps** (at least 3 each): e.g., only one location, no weather-forecast inputs, trend drift across years, uncertainty estimates, etc.

**Q8.3 (2 marks) – Reproducibility.** Save your best model, the scaler (`joblib.dump`), and list the versions of the libraries you used. Restart the kernel and **Run All** — the notebook must run top to bottom without error.

**Executive Summary:**


**Limitations:**


**Next steps:**

In [ ]:
# Reproducibility info
import sys, sklearn
print("Python", sys.version.split()[0], "| TF", tf.__version__, "| numpy", np.__version__,
      "| pandas", pd.__version__, "| sklearn", sklearn.__version__)